# 04 — Customer Segmentation: RFM Methodology & Prototype (Issue #5)

This notebook validates the agreed RFM methodology on `data/processed/cleaned_retail.csv`, demonstrates edge cases, and exports `data/processed/customer_segments.csv`.

**Core rules:** Recency = days since latest purchase; Frequency = distinct invoices; Monetary = sum of `Quantity × UnitPrice`; reference date = one day after the latest valid identified-customer purchase.

In [1]:
from pathlib import Path
import sys
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Locate repository root whether Jupyter starts in repo root or notebooks/.
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root containing src/ and data/.")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_engineering import (
    audit_rfm_input,
    calculate_rfm,
    prepare_rfm_transactions,
)

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_retail.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "customer_segments.csv"
DATA_PATH.relative_to(PROJECT_ROOT)

PosixPath('data/processed/cleaned_retail.csv')

## 1. Load processed retail data

`cleaned_retail.csv` is the production input. RFM performs defensive validation again because customer segmentation has one additional rule: anonymous `Guest` rows cannot represent a real customer.

In [2]:
retail = pd.read_csv(
    DATA_PATH,
    dtype={"InvoiceNo": "string", "CustomerID": "string"},
    low_memory=False,
)
print("shape:", retail.shape)
retail.head()

shape: (524878, 10)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,YearMonth
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010-12
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010-12
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12


In [3]:
audit = audit_rfm_input(retail)
pd.Series(audit, name="rows")

input_rows                     524878
cancelled_rows                      0
missing_customer_rows               0
anonymous_customer_rows        132186
nonpositive_quantity_rows           0
nonpositive_unit_price_rows         0
invalid_invoice_date_rows           0
exact_duplicate_rows                0
Name: rows, dtype: int64

## 2. Calculate RFM on the official cleaned dataset

The function derives the reference date automatically from valid identified-customer purchases and returns scoring metadata so the exact run thresholds are auditable.

In [4]:
customer_segments, metadata = calculate_rfm(retail, return_metadata=True)

summary = pd.Series({
    "valid_transaction_rows": metadata["valid_transaction_rows"],
    "customers": metadata["customers"],
    "invoices": metadata["invoices"],
    "last_valid_purchase": metadata["last_valid_purchase"],
    "reference_date": metadata["reference_date"],
})
summary

valid_transaction_rows                 392692
customers                                4338
invoices                                18532
last_valid_purchase       2011-12-09 12:50:00
reference_date            2011-12-10 00:00:00
dtype: object

In [5]:
thresholds = pd.DataFrame(metadata["scoring"]).T[
    ["method", "q20", "q40", "q60", "q80", "n_unique", "higher_is_better"]
]
thresholds

,method,q20,q40,q60,q80,n_unique,higher_is_better
Recency,quantile,13.80,33.00,72.00,180.00,304,False
Frequency,quantile,1.00,2.00,3.00,6.00,59,True
Monetary,quantile,249.34,487.41,933.35,"2,055.05",4256,True


In [6]:
customer_segments.head(10)

,CustomerID,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_score,RFM_total,Segment,LastPurchaseDate,ReferenceDate
0,14646,2,73,"280,206.02",5,5,5,555,15,Champions,2011-12-08,2011-12-10
1,18102,1,60,"259,657.30",5,5,5,555,15,Champions,2011-12-09,2011-12-10
2,17450,9,46,"194,390.79",5,5,5,555,15,Champions,2011-12-01,2011-12-10
3,14911,2,201,"143,711.17",5,5,5,555,15,Champions,2011-12-08,2011-12-10
4,14156,10,55,"117,210.08",5,5,5,555,15,Champions,2011-11-30,2011-12-10
5,17511,3,31,"91,062.38",5,5,5,555,15,Champions,2011-12-07,2011-12-10
6,16684,5,28,"66,653.56",5,5,5,555,15,Champions,2011-12-05,2011-12-10
7,14096,5,17,"65,164.79",5,5,5,555,15,Champions,2011-12-05,2011-12-10
8,13694,4,50,"65,039.62",5,5,5,555,15,Champions,2011-12-06,2011-12-10
9,15311,1,91,"60,632.75",5,5,5,555,15,Champions,2011-12-09,2011-12-10


## 3. Segment distribution

In [7]:
segment_counts = (
    customer_segments["Segment"]
    .value_counts()
    .rename_axis("Segment")
    .to_frame("Customers")
)
segment_counts["Share_%"] = (segment_counts["Customers"] / len(customer_segments) * 100).round(2)
segment_counts

,Customers,Share_%
Segment,,
Hibernating,954,21.99
Champions,911,21.00
At Risk,773,17.82
Others,599,13.81
Potential Loyalists,484,11.16
Loyal Customers,381,8.78
New Customers,236,5.44


## 4. Acceptance sanity checks on production data

In [8]:
valid = prepare_rfm_transactions(retail)

assert metadata["reference_date"] == metadata["last_valid_purchase"].normalize() + pd.Timedelta(days=1)
assert customer_segments["CustomerID"].nunique() == len(customer_segments)
assert "Guest" not in set(customer_segments["CustomerID"].astype(str))
assert (valid["Quantity"] > 0).all()
assert (valid["UnitPrice"] > 0).all()
assert not valid["InvoiceNo"].astype(str).str.startswith("C").any()
assert valid["CustomerID"].notna().all()
assert customer_segments[["R_score", "F_score", "M_score"]].apply(lambda s: s.between(1, 5).all()).all()
assert customer_segments["Recency"].min() >= 1

print("Production sanity checks passed.")

Production sanity checks passed.


## 5. Explicit prototype edge cases

The small synthetic sample demonstrates the exact cases listed in Issue #5: multiple lines on one invoice, one vs. multiple orders, cancellations, missing/anonymous customers, invalid rows, duplicate rows and tied RFM values.

In [9]:
sample = pd.DataFrame([
    ["10001", "A", 1, "2011-12-01 10:00", 10.0, "1"],
    ["10001", "B", 2, "2011-12-01 10:00",  5.0, "1"],
    ["10002", "C", 1, "2011-12-05 10:00", 20.0, "1"],
    ["10003", "A", 1, "2011-12-09 09:00", 10.0, "2"],
    ["10004", "A", 2, "2011-12-07 09:00", 10.0, "3"],
    ["10005", "A", 2, "2011-12-07 09:00", 10.0, "4"],
    ["C10006", "A", -1, "2011-12-08 09:00", 10.0, "5"],
    ["10007", "A", 1, "2011-12-08 09:00", 10.0, None],
    ["10008", "A", 1, "2011-12-08 09:00", 10.0, "Guest"],
    ["10009", "A", 0, "2011-12-08 09:00", 10.0, "6"],
    ["10010", "A", 1, "2011-12-08 09:00", 0.0, "7"],
    ["10011", "D", 1, "2011-12-06 09:00", 15.0, "8"],
    ["10011", "D", 1, "2011-12-06 09:00", 15.0, "8"],
], columns=["InvoiceNo", "StockCode", "Quantity", "InvoiceDate", "UnitPrice", "CustomerID"])

sample_rfm, sample_meta = calculate_rfm(
    sample, reference_date="2011-12-10", return_metadata=True
)
sample_rfm.sort_values("CustomerID")

,CustomerID,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_score,RFM_total,Segment,LastPurchaseDate,ReferenceDate
0,1,5,2,40.00,1,5,5,155,11,At Risk,2011-12-05,2011-12-10
3,2,1,1,10.00,5,1,1,511,7,New Customers,2011-12-09,2011-12-10
1,3,3,1,20.00,4,1,3,413,8,New Customers,2011-12-07,2011-12-10
2,4,3,1,20.00,4,1,3,413,8,New Customers,2011-12-07,2011-12-10
4,8,4,1,15.00,2,1,2,212,5,Hibernating,2011-12-06,2011-12-10


In [10]:
c1 = sample_rfm.loc[sample_rfm["CustomerID"].eq("1")].iloc[0]
c2 = sample_rfm.loc[sample_rfm["CustomerID"].eq("2")].iloc[0]
c3 = sample_rfm.loc[sample_rfm["CustomerID"].eq("3")].iloc[0]
c4 = sample_rfm.loc[sample_rfm["CustomerID"].eq("4")].iloc[0]
c8 = sample_rfm.loc[sample_rfm["CustomerID"].eq("8")].iloc[0]

assert c1["Frequency"] == 2                 # two invoices, not three product lines
assert c1["Monetary"] == 40.0
assert c2["Recency"] == 1                   # purchase on day before reference date
assert set(sample_rfm["CustomerID"]) == {"1", "2", "3", "4", "8"}
assert c8["Frequency"] == 1 and c8["Monetary"] == 15.0  # duplicate removed
assert tuple(c3[["R_score", "F_score", "M_score"]]) == tuple(c4[["R_score", "F_score", "M_score"]])

print("Synthetic edge-case checks passed.")

Synthetic edge-case checks passed.


## 6. Export `customer_segments.csv`

In [11]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
customer_segments.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(customer_segments):,} customers -> {OUTPUT_PATH.relative_to(PROJECT_ROOT)}")

Saved 4,338 customers -> data/processed/customer_segments.csv


## 7. Issue #5 acceptance checklist

- [x] RFM formulas and reference-date rule are explicit.
- [x] Cancelled invoices, missing/anonymous CustomerID, invalid values and duplicates have explicit rules.
- [x] R/F/M score direction is defined on a 1–5 scale.
- [x] Quantile ties / low-cardinality values have a deterministic fallback.
- [x] Segment labels and precedence thresholds are documented.
- [x] Prototype validates one/multiple orders, multi-line invoices and equal RFM values.
- [x] Production run succeeds on `cleaned_retail.csv`.
- [x] `customer_segments.csv` is exported and ready for downstream analysis.